# Qwen2.5-0.5B-Instruct Conversational Fine-Tuning with LoRA (Colab)

This notebook fine-tunes Qwen/Qwen2.5-0.5B-Instruct using supervised fine-tuning (SFT) and a LoRA adapter.

***Target environment:*** Google Colab with a CUDA GPU, such as an NVIDIA T4.

## ***Important***
- This notebook uses standard LoRA, not QLoRA.
- It uses a small example dataset for experimentation.
- LoRA trains a small number of additional parameters instead of updating all the model weights.
- The notebook uses FP16 precision to suit an NVIDIA T4 GPU.
- Run the cells from top to bottom.
- If Colab asks you to restart the runtime after installing packages, restart it and rerun the cells from the beginning.
- The sample dataset is only for testing the training pipeline. Use a task-specific dataset for meaningful results.

# 1. Check the Colab GPU
This section checks whether a CUDA-compatible GPU is available and displays its name. A GPU is recommended for faster fine-tuning.

In [1]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU not detected. In Colab, choose "
        "Runtime > Change runtime type > T4 GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA available:", torch.cuda.is_available())
print("Training dtype: float16")

GPU: Tesla T4
CUDA available: True
Training dtype: float16


# 2. Install the required libraries
This section installs the Python libraries required for model loading, dataset preparation, LoRA configuration, and supervised fine-tuning.

The libraries include Transformers, Datasets, PEFT, TRL, and Accelerate.

In [2]:
%pip install -q transformers datasets peft trl accelerate

#3. Verify the installed libraries
This section confirms that the required libraries are installed and that the GPU remains available after installation.

In [3]:
import torch
import transformers
import datasets
import peft
import trl
import accelerate

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("Accelerate:", accelerate.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.11.0+cu130
Transformers: 5.20.2
Datasets: 4.8.5
PEFT: 0.21.2
TRL: 1.16.0
Accelerate: 1.15.0
CUDA available: True


# 4. Load Qwen2.5-0.5B-Instruct
This section downloads the pretrained model and tokenizer from Hugging Face.

The tokenizer converts text into tokens, while the language model generates responses. FP16 precision reduces memory use compared with full-precision FP32 weights.

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16
)

model.config.use_cache = False

print("Loaded model:", MODEL_NAME)
print("Tokenizer has chat template:", tokenizer.chat_template is not None)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded model: Qwen/Qwen2.5-0.5B-Instruct
Tokenizer has chat template: True


# 5. Prepare the training dataset
This section creates a small dataset containing example questions and answers. It is intended to verify that the training workflow works correctly.

For a conversational model, examples should ideally follow the model's chat format. Here, we use the tokenizer's chat template to format each conversation.

In [5]:
from datasets import Dataset

conversations = [
    [
        {"role": "user", "content": "What is Python?"},
        {"role": "assistant", "content": "Python is a programming language."}
    ],
    [
        {"role": "user", "content": "What is SQL?"},
        {"role": "assistant", "content": "SQL is used to query and manage relational databases."}
    ],
    [
        {"role": "user", "content": "What is a variable?"},
        {"role": "assistant", "content": "A variable stores a value in a program."}
    ],
    [
        {"role": "user", "content": "What is machine learning?"},
        {"role": "assistant", "content": "Machine learning enables computers to learn patterns from data."}
    ],
]

def format_conversation(messages):
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

dataset = Dataset.from_dict({
    "text": [
        format_conversation(messages)
        for messages in conversations
    ] * 5
})

print("Training examples:", len(dataset))
print(dataset[0]["text"])

Training examples: 20
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What is Python?<|im_end|>
<|im_start|>assistant
Python is a programming language.<|im_end|>



# Configure LoRA
LoRA (Low-Rank Adaptation) adds trainable adapter parameters to selected layers of the model. It reduces the number of parameters that need to be trained and is suitable for learning fine-tuning on a limited GPU budget.

In [6]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

print("LoRA configuration created.")

LoRA configuration created.


# 7. Configure supervised fine-tuning
This section sets the training parameters, including the output directory, number of epochs, batch size, learning rate, and sequence length.

For the first experiment, one epoch and a small batch size help keep the run manageable.

In [7]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="/content/qwen25-finetuned",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    logging_steps=1,
    save_strategy="epoch",
    report_to="none",
    max_length=256,
    fp16=True,
    gradient_checkpointing=True,
    dataset_text_field="text"
)

print("Training configuration created.")

Training configuration created.


In [9]:
%pip install -q -U "torchao>=0.16.0"

# 8. Start fine-tuning
This section creates the trainer and begins training the LoRA adapter on the prepared dataset. Training loss is logged as the run progresses.

In [10]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=lora_config
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Step,Training Loss
1,4.987389
2,5.053868
3,5.003185
4,4.793983
5,4.613535


TrainOutput(global_step=5, training_loss=4.890392017364502, metrics={'train_runtime': 20.6417, 'train_samples_per_second': 0.969, 'train_steps_per_second': 0.242, 'total_flos': 1949258446080.0, 'train_loss': 4.890392017364502, 'epoch': 1.0})

# 9. Save the fine-tuned adapter
This section saves the trained LoRA adapter and tokenizer. The adapter contains the learned changes; the original base model is still required when loading it for inference.

In [11]:
ADAPTER_PATH = "/content/qwen25-lora-adapter"

trainer.model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

print("Adapter saved to:", ADAPTER_PATH)

Adapter saved to: /content/qwen25-lora-adapter


# 10. Test the fine-tuned model
This section sends a question to the fine-tuned model and prints its response. It checks that the adapter can be used for inference after training.

In [12]:
trainer.model.eval()

messages = [
    {"role": "user", "content": "What is Python?"}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(trainer.model.device)

with torch.no_grad():
    outputs = trainer.model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

print(tokenizer.decode(new_tokens, skip_special_tokens=True))

Python is a high-level programming language that was developed by Guido van Rossum and first released in 1991. It has gained widespread popularity for its simplicity, readability, and versatility.

Key features of Python include:

1. **Ease of Use**: Its syntax is designed to be intuitive


# 11. Download the fine-tuned adapter
This section packages the saved adapter into a ZIP file for downloading from Google Colab.

In [13]:
!zip -r /content/qwen25-lora-adapter.zip /content/qwen25-lora-adapter

  adding: content/qwen25-lora-adapter/ (stored 0%)
  adding: content/qwen25-lora-adapter/tokenizer.json (deflated 81%)
  adding: content/qwen25-lora-adapter/tokenizer_config.json (deflated 59%)
  adding: content/qwen25-lora-adapter/README.md (deflated 65%)
  adding: content/qwen25-lora-adapter/chat_template.jinja (deflated 71%)
  adding: content/qwen25-lora-adapter/adapter_model.safetensors (deflated 10%)
  adding: content/qwen25-lora-adapter/adapter_config.json (deflated 60%)
